<a href="https://colab.research.google.com/github/DJChallas/group1/blob/main/Group1_mod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [213]:
import pandas as pd

# Load the illinois.csv file into a pandas DataFrame
df = pd.read_csv('illinois.csv')

# Display the first 5 rows of the DataFrame
print(df.head())

   Area Number                       Industry Title Series Code     Jan-90  \
0            0                  Total Nonfarm         00-000000  5,270,100   
1            0                               Mining   10-000000     18,800   
2            0                   Construction         20-000000    230,700   
3            0                   Manufacturing        30-000000    923,700   
4            0   Trade, Transportation, & Utilities   40-000000  1,130,700   

      Feb-90     Mar-90     Apr-90     May-90     Jun-90     Jul-90  ...  \
0  5,279,100  5,291,300  5,282,100  5,289,600  5,299,200  5,303,300  ...   
1     18,800     18,600     18,300     18,100     18,300     18,300  ...   
2    230,500    229,400    227,000    224,600    223,000    224,400  ...   
3    927,300    927,200    923,200    924,100    922,600    918,100  ...   
4  1,131,600  1,132,600  1,131,600  1,131,600  1,133,900  1,131,700  ...   

      Nov-25     Dec-25     Jan-26     Feb-26     Mar-26     Apr-26  \
0  

### Illinois Industry Employment Trends (Interactive Plotly Visualization)

In [214]:
import pandas as pd
import re
import os
import plotly.express as px

# 1. Direct path lookup for the dataset to define melted_df
csv_paths = ['illinois.csv', '../illinois.csv', '/content/illinois.csv', '/illinois.csv']
df_local = None

for path in csv_paths:
    if os.path.exists(path):
        df_local = pd.read_csv(path)
        break

if df_local is None:
    if 'df' in globals():
        df_local = globals()['df'].copy()
    else:
        raise FileNotFoundError("Could not locate 'illinois.csv' file or loaded 'df' variable in memory.")

# 2. Process and filter
df_local.columns = df_local.columns.str.strip()
industries_df = df_local[df_local['Industry Title'].str.strip() != 'Total Nonfarm'].copy()

month_year_pattern = re.compile(r'^[A-Za-z]{3}-\d{2}$', re.IGNORECASE)
potential_month_cols = [col for col in industries_df.columns if month_year_pattern.match(col)]
try:
    start_col_index = potential_month_cols.index('Feb-90')
    selected_month_cols = potential_month_cols[start_col_index:]
except ValueError:
    selected_month_cols = potential_month_cols

plot_df = industries_df[['Industry Title'] + selected_month_cols]
melted_df = plot_df.melt(id_vars=['Industry Title'], var_name='Month', value_name='Employment')
melted_df['Employment'] = melted_df['Employment'].astype(str).str.replace(',', '', regex=False)
melted_df['Employment'] = pd.to_numeric(melted_df['Employment'], errors='coerce')
melted_df['Month'] = pd.to_datetime(melted_df['Month'], format='%b-%y')
melted_df.dropna(subset=['Employment'], inplace=True)

# 3. Illinois Industry Employment Trends (Muted Color Scheme)
fig_trends_muted = px.line(
    melted_df,
    x='Month',
    y='Employment',
    color='Industry Title',
    title='Illinois Industry Employment Trends (Feb 1990 - Aug 2026)',
    labels={'Month': 'Date', 'Employment': 'Employment'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_trends_muted.update_layout(
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    paper_bgcolor='white',
    plot_bgcolor='white',
    title_x=0.5
)

fig_trends_muted.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_trends_muted.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

fig_trends_muted.show()

In [215]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# Fallback check: If melted_df_sorted is not in memory, calculate it from melted_df
if 'melted_df_sorted' not in globals():
    if 'melted_df' in globals():
        melted_df_sorted = globals()['melted_df'].sort_values(by=['Industry Title', 'Month']).copy()
        melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()
        globals()['melted_df_sorted'] = melted_df_sorted
    else:
        raise NameError("Base DataFrame 'melted_df' is not defined in memory. Please run the data preparation cells first.")

# Fallback check: If recession_dips_df_strict is not in memory, calculate it on the fly
if 'recession_dips_df_strict' in globals():
    dips_df = globals()['recession_dips_df_strict']
else:
    # Identify 5th percentile strict dips using melted_df_sorted
    negative_changes_per_industry_strict = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]
    dip_thresholds_strict = negative_changes_per_industry_strict.groupby('Industry Title')['Employment_Change'].quantile(0.05)

    significant_dips_list_strict = []
    for industry, threshold in dip_thresholds_strict.items():
        industry_dips_strict = melted_df_sorted[
            (melted_df_sorted['Industry Title'] == industry) &
            (melted_df_sorted['Employment_Change'] <= threshold)
        ].copy()
        significant_dips_list_strict.append(industry_dips_strict)

    if significant_dips_list_strict:
        dips_df = pd.concat(significant_dips_list_strict)
    else:
        dips_df = pd.DataFrame(columns=['Industry Title', 'Month', 'Employment', 'Employment_Change'])

    dips_df = dips_df[['Industry Title', 'Month', 'Employment_Change']]
    dips_df.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)
    dips_df['Abs_Dip_Magnitude'] = dips_df['Dip_Magnitude'].abs()
    # Save it globally so it's defined moving forward
    globals()['recession_dips_df_strict'] = dips_df

# Create the base line plot for overall employment trends using the Safe color sequence
fig_combined = px.line(
    melted_df,
    x='Month',
    y='Employment',
    color='Industry Title',
    title='Illinois Industry Employment Trends with 5th Percentile Dips Highlighted',
    labels={'Month': 'Date', 'Employment': 'Employment'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

# Filter to only include industries present in melted_df
dips_to_overlay = dips_df[dips_df['Industry Title'].isin(melted_df['Industry Title'].unique())]

# Loop through each industry to add scatter markers for dips
for industry in dips_to_overlay['Industry Title'].unique():
    industry_dips = dips_to_overlay[dips_to_overlay['Industry Title'] == industry]
    # Get the corresponding employment value from melted_df for each dip month
    employment_at_dip = []
    for _, row in industry_dips.iterrows():
        matching_employment = melted_df[(melted_df['Industry Title'] == industry) & (melted_df['Month'] == row['Month'])]['Employment']
        if not matching_employment.empty:
            employment_at_dip.append(matching_employment.iloc[0])
        else:
            employment_at_dip.append(None)

    fig_combined.add_trace(
        go.Scatter(
            x=industry_dips['Month'],
            y=employment_at_dip,
            mode='markers',
            marker=dict(symbol='triangle-down', size=10, color='red', line=dict(width=1, color='DarkRed')),
            name=f'{industry} Dip',
            hoverinfo='text',
            hovertext=industry_dips.apply(lambda row: f"Industry: {row['Industry Title']}<br>Date: {row['Month'].strftime('%Y-%m')}<br>Dip Magnitude: {row['Dip_Magnitude']:,}", axis=1),
            showlegend=False
        )
    )

# Update layout for better readability
fig_combined.update_layout(
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    title_x=0.5
)

# Add grid for readability
fig_combined.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_combined.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

fig_combined.show()

In [216]:
import pandas as pd
import re
import numpy as np
import plotly.express as px
import os

# 1. Direct path lookup for the dataset
csv_paths = ['illinois.csv', '../illinois.csv', '/content/illinois.csv', '/illinois.csv']
df_local = None

for path in csv_paths:
    if os.path.exists(path):
        df_local = pd.read_csv(path)
        break

if df_local is None:
    if 'df' in globals():
        df_local = globals()['df'].copy()
    else:
        raise FileNotFoundError("Could not locate 'illinois.csv' file or loaded 'df' variable in memory. Please run the first cell to load the data.")

# 2. Process and filter
df_local.columns = df_local.columns.str.strip()
industries_df = df_local[df_local['Industry Title'].str.strip() != 'Total Nonfarm'].copy()

month_year_pattern = re.compile(r'^[A-Za-z]{3}-\d{2}$', re.IGNORECASE)
potential_month_cols = [col for col in industries_df.columns if month_year_pattern.match(col)]
try:
    start_col_index = potential_month_cols.index('Feb-90')
    selected_month_cols = potential_month_cols[start_col_index:]
except ValueError:
    selected_month_cols = potential_month_cols

plot_df = industries_df[['Industry Title'] + selected_month_cols]
melted_df = plot_df.melt(id_vars=['Industry Title'], var_name='Month', value_name='Employment')
melted_df['Employment'] = melted_df['Employment'].astype(str).str.replace(',', '', regex=False)
melted_df['Employment'] = pd.to_numeric(melted_df['Employment'], errors='coerce')
melted_df['Month'] = pd.to_datetime(melted_df['Month'], format='%b-%y')
melted_df.dropna(subset=['Employment'], inplace=True)

melted_df_sorted = melted_df.sort_values(by=['Industry Title', 'Month'])
melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()

negative_changes = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]

# Stating and confirming the use of the 2nd percentile threshold
dip_thresholds_very_strict = negative_changes.groupby('Industry Title')['Employment_Change'].quantile(0.02)

significant_dips_list_very_strict = []
for industry, threshold in dip_thresholds_very_strict.items():
    industry_dips_very_strict = melted_df_sorted[
        (melted_df_sorted['Industry Title'] == industry) &
        (melted_df_sorted['Employment_Change'] <= threshold)
    ].copy()
    significant_dips_list_very_strict.append(industry_dips_very_strict)

recession_dips_df_very_strict = pd.concat(significant_dips_list_very_strict)
recession_dips_df_very_strict.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)
recession_dips_df_very_strict['Abs_Dip_Magnitude'] = recession_dips_df_very_strict['Dip_Magnitude'].abs()

# 3. Handle symmetric log transformation manually for clean plotting of negative values
recession_dips_df_very_strict['Log_Dip_Coordinate'] = -np.log10(recession_dips_df_very_strict['Abs_Dip_Magnitude'])
max_dip = recession_dips_df_very_strict['Abs_Dip_Magnitude'].max()

ticks = [10**i for i in range(1, int(np.ceil(np.log10(max_dip))) + 1)]
ticks = [t for t in ticks if t <= max_dip]
tick_positions = [-np.log10(t) for t in ticks]
tick_labels = [f'-{t:,}' for t in ticks]

# 4. Interactive Plot using Safe muted layout styling (markers + muted dashed lines)
fig_downward_log = px.line(
    recession_dips_df_very_strict,
    x='Month',
    y='Log_Dip_Coordinate',
    color='Industry Title',
    title='Downward Visualized Monthly Employment Dips - 2nd Percentile (Symmetric Log Scale)',
    labels={
        'Month': 'Date',
        'Log_Dip_Coordinate': 'Contraction Severity (2nd Percentile Dips)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_data={'Month': '|%Y-%m-%d', 'Industry Title': True, 'Dip_Magnitude': ':, .0f', 'Log_Dip_Coordinate': False}
)

# Update traces to include markers and muted dashed lines
fig_downward_log.update_traces(
    mode='lines+markers',
    line=dict(dash='dash', width=1.5),
    marker=dict(size=8, opacity=0.8, line=dict(width=1, color='DimGray'))
)

fig_downward_log.update_layout(
    title_x=0.5,
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_downward_log.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_downward_log.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='#E8E8E8',
    tickvals=tick_positions,
    ticktext=tick_labels,
    title_text='Monthly Dip Magnitude (2nd Percentile, Log Scale)'
)

fig_downward_log.show()

### Methodology & Parameters for Very Extreme 2nd Percentile Dips

To map and compare the severe contraction events across different sectors, this visualization uses the following structured quantitative approach:

1. **Time Frame**:
   - The analysis spans the full duration of our historical dataset: **February 1990 to August 2026**.

2. **Calculation Method**:
   - **Month-over-Month (MoM) Changes**: First, the monthly employment change is calculated for each industry title:
     $$\Delta E_{i, t} = Employment_{i, t} - Employment_{i, t-1}$$
   - **Isolating Contractions**: The dataset is filtered to only include negative monthly changes (where $\Delta E_{i, t} < 0$).
   - **Threshold Determination (2nd Percentile)**: For each industry, we compute a customized "severe threshold" equivalent to its **2nd percentile (0.02 quantile)** of negative changes. This isolates only the absolute worst 2% of monthly job contraction events unique to that sector's historical volatility.
   - **Filtering**: Only months where the contraction severity equals or exceeds this 2nd percentile threshold are retained for plotting.

3. **Visual Representation (Logarithmic Scaling)**:
   - Because massive anomalies (such as the 2020 COVID-19 shock) dwarf normal business cycle contractions, we visualize the absolute value of the job loss magnitude (`Abs_Dip_Magnitude`) on a **logarithmic (Log10) y-axis**.
   - This logarithmic transformation compresses the vertical scale, allowing you to clearly compare smaller historical dips (on the scale of hundreds or thousands of jobs) side-by-side with massive multi-thousand-job displacements.

In [217]:
# Ensure melted_df_sorted is sorted and has the 'Employment_Change' column computed
melted_df_sorted = melted_df_sorted.sort_values(by=['Industry Title', 'Month'])
if 'Employment_Change' not in melted_df_sorted.columns:
    melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()

# Calculate the 2nd percentile of negative Employment_Change for each industry
# Only consider industries with at least some negative changes to calculate percentile
negative_changes_per_industry_very_strict = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]

# Ensure there are negative changes before calculating quantiles
if not negative_changes_per_industry_very_strict.empty:
    dip_thresholds_very_strict = negative_changes_per_industry_very_strict.groupby('Industry Title')['Employment_Change'].quantile(0.02)
else:
    dip_thresholds_very_strict = pd.Series([], dtype='float64') # Empty series if no negative changes

# Identify significant dips based on the calculated very-strict thresholds
significant_dips_list_very_strict = []
for industry, threshold in dip_thresholds_very_strict.items():
    industry_dips_very_strict = melted_df_sorted[
        (melted_df_sorted['Industry Title'] == industry) &
        (melted_df_sorted['Employment_Change'] <= threshold)
    ].copy()
    significant_dips_list_very_strict.append(industry_dips_very_strict)

# Concatenate all identified significant dips into a single DataFrame
if significant_dips_list_very_strict:
    recession_dips_df_very_strict = pd.concat(significant_dips_list_very_strict)
else:
    recession_dips_df_very_strict = pd.DataFrame(columns=['Industry Title', 'Month', 'Employment', 'Employment_Change'])

# Select relevant columns for the final DataFrame and rename for clarity
recession_dips_df_very_strict = recession_dips_df_very_strict[['Industry Title', 'Month', 'Employment_Change']]
recession_dips_df_very_strict.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)

# Add a column for absolute dip magnitude for log scale visualization
recession_dips_df_very_strict['Abs_Dip_Magnitude'] = recession_dips_df_very_strict['Dip_Magnitude'].abs()

print("DataFrame of Very Extreme Significant Monthly Employment Dips (2nd Percentile Threshold):")
display(recession_dips_df_very_strict.sort_values(by=['Month', 'Industry Title']))

DataFrame of Very Extreme Significant Monthly Employment Dips (2nd Percentile Threshold):


,Industry Title,Month,Dip_Magnitude,Abs_Dip_Magnitude
121,Mining,1991-01-01,-700.0,700.0
244,Manufacturing,1991-12-01,-10900.0,10900.0
255,Manufacturing,1992-01-01,-16500.0,16500.0
329,Government,1992-07-01,-14900.0,14900.0
451,Mining,1993-07-01,-1300.0,1300.0
522,Financial Activities,1994-01-01,-7700.0,7700.0
539,Mining,1994-03-01,-600.0,600.0
621,Financial Activities,1994-10-01,-3600.0,3600.0
623,Private Education and Health Services,1994-10-01,-9900.0,9900.0
686,Information,1995-04-01,-3400.0,3400.0


In [218]:
import pandas as pd
import plotly.express as px

# Dynamic Check: Rebuild correlation_matrix if it is missing from global memory
if 'correlation_matrix' not in globals():
    if 'melted_df' in globals():
        pivoted_df = globals()['melted_df'].pivot_table(index='Month', columns='Industry Title', values='Employment')
        correlation_matrix = pivoted_df.corr()
        globals()['correlation_matrix'] = correlation_matrix
    else:
        raise NameError("Base dataset 'melted_df' is not defined in memory. Please run your data loading and preparation cells first.")

# 1. Plot the Interactive Correlation Matrix Heatmap
fig_corr = px.imshow(
    correlation_matrix,                 # pandas DataFrame
    labels=dict(x='Industry', y='Industry', color='Correlation'),
    text_auto='.2f',                    # show values like annot=True, fmt=".2f"
    color_continuous_scale='RdBu_r',    # close to seaborn 'coolwarm' (reversed)
    zmin=-1, zmax=1                     # symmetric range for correlations
)

fig_corr.update_layout(
    title=dict(
        text='Correlation Matrix of Illinois Industry Employment Trends',
        x=0.5, xanchor='center'         # center title to match your other figs
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=1200, height=800,
    margin=dict(l=100, r=60, t=80, b=120)
)

fig_corr.update_xaxes(
    tickangle=45,                       # rotate x labels like your seaborn example
    side='bottom'                       # keep ticks at bottom
)
# y-axis labels are horizontal by default (rotation 0), like seaborn

# Optional: smaller annotation text and cleaner hover
fig_corr.update_traces(
    textfont_size=10,
    hovertemplate='Industry i: %{y}<br>Industry j: %{x}<br>r = %{z:.2f}<extra></extra>'
)

fig_corr.show()

### Methodology & Mathematical Process: Industry Correlation Matrix

To map and understand how employment trends move in tandem across different sectors in Illinois, we construct a pairwise **Pearson Correlation Matrix**. This mathematical process measures the strength and direction of linear relationships between the monthly employment levels of each industry over the full historical span (February 1990 to August 2026).

#### Step 1: Reshaping the Data (Pivoting)
The historical "long" format dataset (`melted_df`) is pivoted into a "wide" representation where:
* Each column $X$ represents an **individual industry** (e.g., *Manufacturing*, *Financial Activities*).
* Each row $t$ represents a specific **historical month** ($t = 1, 2, \dots, n$, where $n = 439$ months).

#### Step 2: Evaluating Pairwise Pearson Correlation Coefficients
For any two selected sectors, Industry $X$ and Industry $Y$, the correlation coefficient $r_{xy}$ is calculated using the Pearson product-moment correlation formula:

$$r_{xy} = \frac{\sum_{t=1}^{n} (x_t - \bar{x})(y_t - \bar{y})}{\sqrt{\sum_{t=1}^{n} (x_t - \bar{x})^2} \sqrt{\sum_{t=1}^{n} (y_t - \bar{y})^2}}$$

Where:
* $x_t$ and $y_t$ are the employment figures for Industry $X$ and Industry $Y$ respectively in month $t$.
* $\bar{x}$ and $\bar{y}$ are the historical mean averages of employment for those respective industries over the entire $n$-month timeframe.

#### Step 3: Equation Components Explained
* **Numerator (Sample Covariance):** $\sum_{t=1}^{n} (x_t - \bar{x})(y_t - \bar{y})$ evaluates whether the two industries deviate from their averages in the same direction at the same time. If both industries consistently expand (or contract) together, this term accumulates a large positive value.
* **Denominator (Sample Standard Deviations):** $\sqrt{\sum_{t=1}^{n} (x_t - \bar{x})^2} \sqrt{\sum_{t=1}^{n} (y_t - \bar{y})^2}$ standardizes the relationship by dividing by the product of each industry's internal volatility (variance). This removes scale effects, preventing industries with massive employee bases (e.g., *Trade, Transportation, & Utilities*) from skewing results relative to smaller industries (e.g., *Mining*).

#### Step 4: Interpreting the Scale
The resulting coefficient $r_{xy}$ is scale-invariant and strictly bounded within $[-1, 1]$:
* **$r_{xy} \approx 1.0$ (Strong Positive Correlation):** Indicates that the two industries share closely aligned business cycle forces, growing and contracting in synchronous lockstep.
* **$r_{xy} \approx 0.0$ (No Linear Correlation):** Indicates that growth or contraction trends in Industry $X$ have no linear relationship to employment levels in Industry $Y$.
* **$r_{xy} \approx -1.0$ (Strong Negative Correlation):** Indicates a counter-cyclical relationship, where contractions in Industry $X$ consistently coincide with expansions in Industry $Y$.

### Historical Recession Profiles & Context

Below is the historical context corresponding to the four numbered columns analyzed in the Sector Vulnerability comparison heatmap above:

1. **1990-1991 Recession**: A mild contraction influenced by restrictive monetary policy, the 1990 crude oil price shock following Iraq's invasion of Kuwait, and structural corrections following the 1980s real estate and commercial debt boom.
2. **2001-2003 Dot-Com & 9/11 Shock**: The unwinding of extreme technology stock valuations led to severe contractions in IT, Professional Services, and Corporate Management, exacerbated by immediate physical and financial disruptions after the September 11 terrorist attacks.
3. **2007-2009 Great Recession**: Triggered by the systemic collapse of the subprime mortgage market and subsequent global financial liquidity crisis. This led to persistent, extreme contractions in Construction, Manufacturing, Finance, and retail operations.
4. **2020 COVID-19 Outbreak**: An unprecedented public-health-induced freeze of physical commercial spaces. While brief compared to other crises, it triggered the most catastrophic single-month labor disruptions on record, primarily focused in Leisure and Hospitality and physical retail trade.

### Historical Context & Explanations for Significant Employment Dips

We will map major periods of employment dips in the Illinois dataset (such as those matching the Dot-Com crash, the Great Recession, and the COVID-19 shock) to their historical drivers and create an explanations DataFrame.

In [219]:
import pandas as pd

# Define major economic events in US/Illinois history that correlate with these dips
historical_events = [
    {
        'Period': '1990-1991',
        'Economic Event': 'Early 1990s Recession',
        'Explanation': 'A mild recession caused by restrictive monetary policy, the 1990 oil price shock following Iraq\'s invasion of Kuwait, and structural adjustments after the 1980s real estate boom.'
    },
    {
        'Period': '2001-2003',
        'Economic Event': 'Dot-com Bubble Burst & 9/11 Attacks',
        'Explanation': 'Collapse of tech stock valuations led to widespread job losses in Professional Services, Corporate Management, and Manufacturing, exacerbated by the shock of September 11.'
    },
    {
        'Period': '2007-2009',
        'Economic Event': 'The Great Recession',
        'Explanation': 'Triggered by the subprime mortgage crisis and the collapse of housing markets, leading to severe job cuts in Construction, Manufacturing, Finance, and Retail.'
    },
    {
        'Period': '2020',
        'Economic Event': 'COVID-19 Pandemic Outbreak',
        'Explanation': 'Unprecedented service sector disruption due to public health lockdowns, affecting Leisure and Hospitality, retail trade, and personal services immediately.'
    }
]

# Create the explanation dataframe
explanations_df = pd.DataFrame(historical_events)
display(explanations_df)

,Period,Economic Event,Explanation
0,1990-1991,Early 1990s Recession,A mild recession caused by restrictive monetar...
1,2001-2003,Dot-com Bubble Burst & 9/11 Attacks,Collapse of tech stock valuations led to wides...
2,2007-2009,The Great Recession,Triggered by the subprime mortgage crisis and ...
3,2020,COVID-19 Pandemic Outbreak,Unprecedented service sector disruption due to...


In [220]:
import plotly.express as px
import numpy as np

# Define date filters for the 4 major economic crashes
crashes_filters = {
    '1990-1991 Recession': ('1990-01-01', '1992-06-01'),
    '2001-2003 Dot-Com & 9/11 Shock': ('2001-01-01', '2003-12-01'),
    '2007-2009 Great Recession': ('2007-12-01', '2010-06-01'),
    '2020 COVID-19 Pandemic Outbreak': ('2020-04-01', '2020-11-01')
}

# Generate a log-transformed bar plot for each crash period
for title, (start_date, end_date) in crashes_filters.items():
    crash_data = recession_dips_df_very_strict[
        (recession_dips_df_very_strict['Month'] >= start_date) &
        (recession_dips_df_very_strict['Month'] <= end_date)
    ].copy()

    if not crash_data.empty:
        # Convert negative dip magnitudes to positive values for traditional bottom-up representation
        crash_data['Abs_Dip_Magnitude'] = crash_data['Dip_Magnitude'].abs()

        fig = px.bar(
            crash_data.sort_values(by='Abs_Dip_Magnitude', ascending=False),
            x='Month',
            y='Abs_Dip_Magnitude',
            color='Industry Title',
            title=f'Log-Scaled Severe Industry Dips During the {title}',
            labels={'Abs_Dip_Magnitude': 'Employment Decrease (Jobs Lost)', 'Month': 'Date'},
            barmode='group',
            log_y=True,
            color_discrete_sequence=px.colors.qualitative.Safe
        )
        fig.update_layout(
            yaxis_title='Employment Loss (Jobs Lost - Log Scale)',
            xaxis_title='Month',
            paper_bgcolor='white',
            plot_bgcolor='white',
            title_x=0.5
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
        fig.show()
    else:
        print(f"No extreme 2nd percentile dips recorded in our dataset for the period: {title}")

In [221]:
import numpy as np
import pandas as pd
import plotly.express as px

# Calculate the total absolute dip losses for each industry during each crash window
heatmap_data_log = []

all_industries = recession_dips_df_very_strict['Industry Title'].unique()

for title, (start_date, end_date) in crashes_filters.items():
    for industry in all_industries:
        losses = recession_dips_df_very_strict[
            (recession_dips_df_very_strict['Month'] >= start_date) &
            (recession_dips_df_very_strict['Month'] <= end_date) &
            (recession_dips_df_very_strict['Industry Title'] == industry)
        ]['Dip_Magnitude'].sum()

        actual_loss = losses if not np.isnan(losses) else 0.0
        abs_loss = abs(actual_loss)

        # Apply log10 transformation to the absolute loss to increase color scale variability
        log_loss = np.log10(abs_loss + 1) if abs_loss > 0 else 0.0

        heatmap_data_log.append({
            'Crash': title,
            'Industry': industry.strip(),
            'Log Loss': log_loss,
            'Actual Loss': actual_loss
        })

heatmap_log_df = pd.DataFrame(heatmap_data_log)

# Pivot dataframes for the heatmap matrix values and the hover text annotations
try:
    pivoted_log = heatmap_log_df.pivot(index='Industry', columns='Crash', values='Log Loss').fillna(0)
    pivoted_actual = heatmap_log_df.pivot(index='Industry', columns='Crash', values='Actual Loss').fillna(0)
except ValueError:
    # Fallback if there are duplicates
    pivoted_log = heatmap_log_df.pivot_table(index='Industry', columns='Crash', values='Log Loss', aggfunc='mean').fillna(0)
    pivoted_actual = heatmap_log_df.pivot_table(index='Industry', columns='Crash', values='Actual Loss', aggfunc='sum').fillna(0)

# Create interactive heatmap with log transformed color mapping
fig_log_heatmap = px.imshow(
    pivoted_log,
    labels=dict(x="Economic Crash", y="Industry Title", color="Log-scaled Severity"),
    x=pivoted_log.columns,
    y=pivoted_log.index,
    color_continuous_scale='Reds',
    title='Log-Scaled Comparison Matrix: Sector Vulnerability across Major Crashes'
)

# Update the hover template to show actual job losses instead of the log value
fig_log_heatmap.update_traces(
    customdata=pivoted_actual.values,
    hovertemplate="<b>%{y}</b><br>Crash: %{x}<br>Log Magnitude: %{z:.2f}<br>Actual Job Loss: %{customdata:,.0f}<extra></extra>"
)

fig_log_heatmap.update_layout(
    title={
        'text': 'Log-Scaled Comparison Matrix: Sector Vulnerability across Major Crashes',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_tickangle=-45,
    width=1150,
    height=650,
    coloraxis_colorbar=dict(
        title="Log10(Loss + 1)",
        tickvals=[0, 1, 2, 3, 4, 5],
        ticktext=["0", "10", "100", "1k", "10k", "100k"]
    )
)

fig_log_heatmap.show()

In [222]:
# Strip any whitespace from the column names first to prevent KeyError
df.columns = df.columns.str.strip()
raw_industries = df['Industry Title'].unique()
print("Raw Industry Titles in CSV:")
for ind in raw_industries:
    print(f"- {repr(ind)}")

Raw Industry Titles in CSV:
- 'Total Nonfarm      '
- 'Mining'
- ' Construction      '
- ' Manufacturing     '
- ' Trade, Transportation, & Utilities'
- ' Information       '
- ' Financial Activities'
- ' Professional and Business Services'
- ' Private Education and Health Services'
- ' Leisure and Hospitality'
- ' Other Services    '
- 'Government         '


### Academically Rigorous Spillover Analysis with Log-Differences, Lagged Dependent Variables, and Newey-West HAC Standard Errors

To make our spillover models econometrically defensible, we implement three critical upgrades:
1. **Stationarity & Scale Normalization:** Transform employment $E_t$ into monthly percentage growth rates using log differences: $\Delta \ln(E_t) = \ln(E_t) - \ln(E_{t-1}) \approx \% \Delta E_t$.
2. **Dynamic Specification (AR(1) lag):** Include $\Delta \ln(E_{Follower, t-1})$ to model historical employment inertia and correct serial autocorrelation.
3. **HAC Covariance Estimator:** Use the **Newey-West** correction with a 2-month lag band to ensure standard errors and $p$-values are robust to heteroskedasticity and serial correlation.

In [223]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

# 1. Pivot raw monthly employment levels (instead of pre-computed changes) to calculate percentage changes cleanly
pivoted_employment = melted_df_sorted.pivot(index='Month', columns='Industry Title', values='Employment').fillna(method='ffill')
pivoted_employment.columns = pivoted_employment.columns.str.strip()

# 2. Convert raw employment to Log Differences (Percentage growth rates)
log_returns = np.log(pivoted_employment).diff().dropna() * 100 # Expressed as %

# 3. Apply the recession mask
recession_mask_log = pd.Series(False, index=log_returns.index)
for title, (start_date, end_date) in crashes_filters.items():
    recession_mask_log = recession_mask_log | ((log_returns.index >= start_date) & (log_returns.index <= end_date))

recession_log_returns = log_returns[recession_mask_log].copy()

leader_cols = ['Manufacturing', 'Professional and Business Services', 'Financial Activities']
follower_cols = ['Trade, Transportation, & Utilities', 'Construction', 'Leisure and Hospitality', 'Other Services']

rigorous_spillover_results = []

print("="*115)
print("     ESTIMATING RIGOROUS CROSS-INDUSTRY SPILLOVER MODELS (WITH HAC COVARIANCE & AR(1) CORRECTIONS)")
print("="*115)

for leader in leader_cols:
    for follower in follower_cols:
        if leader in recession_log_returns.columns and follower in recession_log_returns.columns:
            # Define Y (current follower % growth)
            y = recession_log_returns[follower]

            # Define X with Lagged Follower (Y_{t-1}), Current Leader (X_t), and Lagged Leader (X_{t-1})
            X_df = pd.DataFrame({
                'Follower_Lag1': recession_log_returns[follower].shift(1),
                'Leader_Current': recession_log_returns[leader],
                'Leader_Lag1': recession_log_returns[leader].shift(1)
            }).dropna()

            # Align target y with the dropped row from lag operations
            y_aligned = y.loc[X_df.index]
            X_aligned = sm.add_constant(X_df)

            # Fit model using OLS but evaluate with Newey-West HAC robust standard errors
            # maxlags=2 represents a standard lag window for MoM business cycle shifts
            robust_model = sm.OLS(y_aligned, X_aligned).fit(cov_type='HAC', cov_kwds={'maxlags': 2})

            # Run a formal Breusch-Godfrey test for autocorrelation on residuals
            bg_test = acorr_breusch_godfrey(robust_model, nlags=1)
            bg_pvalue = bg_test[1]

            rigorous_spillover_results.append({
                'Leader Industry': leader,
                'Follower Industry': follower,
                'Follower Autoregressive Lag (Beta_Y1)': robust_model.params['Follower_Lag1'],
                'Leader Immediate Coefficient (Beta_X0)': robust_model.params['Leader_Current'],
                'Leader Immediate p-val': robust_model.pvalues['Leader_Current'],
                'Leader Lagged Coefficient (Beta_X1)': robust_model.params['Leader_Lag1'],
                'R-squared': robust_model.rsquared,
                'Durbin-Watson': sm.stats.stattools.durbin_watson(robust_model.resid),
                'Breusch-Godfrey Serial Corr p-val': bg_pvalue
            })

rigorous_spillover_df = pd.DataFrame(rigorous_spillover_results)
display(rigorous_spillover_df)

     ESTIMATING RIGOROUS CROSS-INDUSTRY SPILLOVER MODELS (WITH HAC COVARIANCE & AR(1) CORRECTIONS)


/tmp/ipykernel_5129/3381128967.py:7: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

/tmp/ipykernel_5129/3381128967.py:51: FutureWarning:

acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.

/tmp/ipykernel_5129/3381128967.py:51: FutureWarning:

acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.

/tmp/ipykerne

,Leader Industry,Follower Industry,Follower Autoregressive Lag (Beta_Y1),Leader Immediate Coefficient (Beta_X0),Leader Immediate p-val,Leader Lagged Coefficient (Beta_X1),R-squared,Durbin-Watson,Breusch-Godfrey Serial Corr p-val
0,Manufacturing,"Trade, Transportation, & Utilities",0.386367,1.158596,5.670028e-08,-0.490852,0.778304,1.920364,0.316795
1,Manufacturing,Construction,0.124834,1.203209,8.960514e-08,-0.571537,0.528205,1.972834,0.723882
2,Manufacturing,Leisure and Hospitality,0.501544,5.902463,5.725414e-06,-3.120903,0.735244,2.041368,0.241950
3,Manufacturing,Other Services,0.431230,2.039771,6.414526e-06,-1.130052,0.704353,2.130863,0.102958
4,Professional and Business Services,"Trade, Transportation, & Utilities",0.266832,1.042002,2.268442e-10,-0.367766,0.780027,2.021151,0.333592
5,Professional and Business Services,Construction,0.294568,1.227938,9.458250e-40,-0.777672,0.667606,1.964251,0.773013
6,Professional and Business Services,Leisure and Hospitality,0.388623,5.224928,2.916328e-06,-2.579222,0.743749,1.949492,0.327551
7,Professional and Business Services,Other Services,0.269705,1.812855,2.416013e-06,-0.815840,0.706656,2.154091,0.004370
8,Financial Activities,"Trade, Transportation, & Utilities",0.154244,2.643840,2.730033e-03,-0.839028,0.582833,2.148931,0.001902
9,Financial Activities,Construction,-0.011485,3.360521,9.260688e-07,-0.781417,0.522318,2.079254,0.088616


In [224]:
import pandas as pd

# Construct the Spillover Multiplier Matrix using the rigorous log-difference specifications from rigorous_full_timeline_df
rigorous_spillover_matrix = rigorous_full_timeline_df[[
    'Leader Industry',
    'Follower Industry',
    'Follower Lag Coefficient (Beta_Y1)',
    'Leader Immediate Coefficient (Beta_X0)',
    'Leader Lagged Coefficient (Beta_X1)',
    'R-squared',
    'Durbin-Watson'
]].copy()

# Rename columns for clear presentation
rigorous_spillover_matrix.columns = [
    'Leader Industry (Shock Source)',
    'Follower Industry (Impacted)',
    'Follower Lag (Beta_Y1)',
    'Immediate Multiplier (Beta_X0)',
    'Lagged Multiplier (Beta_X1)',
    'R-squared',
    'Durbin-Watson'
]

print("Rigorous Cross-Industry Job Loss Spillover Multiplier Matrix (Log-Differences %):")
display(rigorous_spillover_matrix)

Rigorous Cross-Industry Job Loss Spillover Multiplier Matrix (Log-Differences %):


,Leader Industry (Shock Source),Follower Industry (Impacted),Follower Lag (Beta_Y1),Immediate Multiplier (Beta_X0),Lagged Multiplier (Beta_X1),R-squared,Durbin-Watson
0,Manufacturing,"Trade, Transportation, & Utilities",0.154958,0.866990,-0.205055,0.568672,2.011418
1,Manufacturing,Construction,-0.109562,1.146393,-0.206841,0.290579,2.013365
2,Manufacturing,Leisure and Hospitality,0.339076,4.063855,-1.822876,0.517956,2.043407
3,Manufacturing,Other Services,0.279916,1.478149,-0.705168,0.504825,2.083166
4,Professional and Business Services,"Trade, Transportation, & Utilities",0.032777,0.706015,-0.065353,0.510008,2.018348
5,Professional and Business Services,Construction,-0.123533,0.992245,-0.137891,0.286373,2.021917
6,Professional and Business Services,Leisure and Hospitality,0.231585,3.297609,-1.143623,0.466197,2.011446
7,Professional and Business Services,Other Services,0.119753,1.183898,-0.348664,0.443490,2.063483
8,Financial Activities,"Trade, Transportation, & Utilities",0.074453,0.748665,-0.188925,0.152114,2.003465
9,Financial Activities,Construction,-0.100671,1.123298,-0.026118,0.102331,2.001962


## Methodology Report: Estimating Illinois December 2030 AI Job Displacement & Cross-Industry Spillover Models

This document details the step-by-step quantitative methodology used to calculate the projected job loss across Illinois industries for December 2030 and our rigorous econometric framework for analyzing historical labor market spillovers.

### 1. Estimating the Baseline Employment (Non-AI Scenario)
To establish a baseline employment scenario for December 2030, we constructed a historical linear trend model for each individual industrial sector:
- **Data Source**: Historical Illinois employment records from `illinois.csv`, spanning from February 1990 to August 2026.
- **Trend Window Selection**: We utilized a modern 10-year rolling window (the last 120 months leading up to August 2026). This filters out outdated structural macroeconomic dynamics and ensures our trend line reflects contemporary post-recessionary growth rates.
- **Linear Modeling**: For each industry, we fit an Ordinary Least Squares (OLS) linear regression model:
  $$E_{base}(t) = \beta_0 + \beta_1 \cdot t$$
  Where $E_{base}(t)$ is the estimated baseline employment, $t$ represents elapsed days since the start of our modern trend window, $\beta_0$ is the intercept, and $\beta_1$ is the daily growth velocity. This baseline was projected to $t = \text{December 1, 2030}$ to generate our non-AI scenario ($6,177,598$ total jobs).

### 2. Formulating the AI Displacement Risk Model
The susceptibility of any given sector to AI displacement relies on task composition, exposure, and human labor dynamics:
- **The AI Impact Equation**:
  $$L_i = (E_i \times S_i) \times (1 - A_i)$$
  Where:
  - $E_i$ (Exposure Score): The proportion of daily tasks in a given sector exposed to AI capability.
  - $S_i$ (Substitution Potential): The rate at which exposed tasks translate into outright automation.
  - $A_i$ (Augmentation Rate): The degree to which AI increases human productivity, serving as a job-preserving multiplier rather than a replacement mechanism.

### 3. Academically Rigorous Cross-Industry Spillover Analysis
To capture how employment contractions in major **Shock Leader** industries transmit to **Follower** industries, we address classical time-series specification errors (such as non-stationarity and serial correlation) with three critical econometric upgrades:

1. **Stationarity & Scale Normalization (Log-Differences)**:
   To prevent scale imbalances and model sector relationships as elasticities, raw monthly employment $E_{j,t}$ is transformed into monthly percentage growth rates using log differences:
   $$y_{j,t} = \Delta \ln(E_{j,t}) \times 100 = \left( \ln(E_{j,t}) - \ln(E_{j,t-1}) \right) \times 100 \approx \% \Delta E_{j,t}$$

2. **Dynamic Specification (AR(1) correction)**:
   Employment trends exhibit severe inertia. To resolve residual serial correlation and eliminate OLS bias, we include the lagged dependent variable (Follower Lag 1) directly in our specification:
   $$y_{Follower, t} = \alpha + \phi y_{Follower, t-1} + \beta_0 y_{Leader, t} + \beta_1 y_{Leader, t-1} + \epsilon_t$$

3. **HAC Covariance Estimator (Newey-West Robust Standard Errors)**:
   To account for remaining heteroskedasticity and residual autocorrelation across historical recessionary windows, we estimate standard errors using the **Newey-West** covariance estimator with a 2-month lag band:
   $$\hat{\Gamma} = \hat{\Gamma}_0 + \sum_{l=1}^{L} \left(1 - \frac{l}{L+1}\right) (\hat{\Gamma}_l + \hat{\Gamma}_l')$$
   This ensures all hypothesis testing, $t$-statistics, and $p$-values are statistically robust and academically defensible.

### Methodology Detail: Walkthrough of the December 2030 Baseline Linear Regression Model

To make our projection methodology completely transparent for the final report, this cell isolates the exact linear regression steps applied to each sector.

Using **Professional and Business Services** as an illustrative example, the code performs the following steps:
1. **Window Selection**: Filters the most recent 120 months (10 years) of historical data to capture modern structural growth velocities.
2. **Time Normalization**: Measures the elapsed time in **Days** from the start of this 120-month window ($x = 0$ corresponds to September 2016).
3. **OLS Regression**: Calculates the slope ($eta_1$, growth per day) and intercept ($eta_0$, base level at start of window).
4. **Extrapolation**: Determines the number of elapsed days between the start of the window and December 1, 2030, and evaluates the model to find the projected baseline.

In [225]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1. Target Sector for demonstration
target_sector = 'Professional and Business Services'

# 2. Ensure the stripped 'Industry_Clean' column exists, then isolate target sector records
melted_df_sorted['Industry_Clean'] = melted_df_sorted['Industry Title'].str.strip()
sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == target_sector].copy()

# 3. Pull the modern 10-year rolling window (last 120 historical months)
recent_data = sector_data.tail(120).copy()

# 4. Define start time of the training window and calculate elapsed days for each data point
start_time = recent_data['Month'].min()
recent_data['Days'] = (recent_data['Month'] - start_time).dt.days

# 5. Extract our features (X_demo = Days) and targets (y_demo = Employment)
X_demo = recent_data[['Days']].values
y_demo = recent_data['Employment'].values

# 6. Instantiate and fit the Scikit-Learn Linear Regression model
model_demo = LinearRegression()
model_demo.fit(X_demo, y_demo)

# 7. Extract the model's coefficients
slope_per_day = model_demo.coef_[0]
slope_per_year = slope_per_day * 365.25
intercept = model_demo.intercept_

# 8. Define target evaluation date and calculate target elapsed days
target_date_dec_2030 = pd.to_datetime('2030-12-01')
target_days_delta = (target_date_dec_2030 - start_time).days

# 9. Perform prediction mathematically and via model.predict()
math_prediction = intercept + (slope_per_day * target_days_delta)
model_prediction = model_demo.predict([[target_days_delta]])[0]

# --- PRINT OUT STEP-BY-STEP MATHEMATICAL SUMMARY FOR REPORT ---
print("="*90)
print(f"LINEAR REGRESSION VERIFICATION: {target_sector.upper()}")
print("="*90)
print(f"Historical Regression Window : {start_time.strftime('%b %Y')} to {recent_data['Month'].max().strftime('%b %Y')} (120 months)")
print(f"Fitted Model Intercept (Beta_0) : {intercept:,.4f} (Expected jobs at window start)")
print(f"Fitted Model Daily Slope (Beta_1): {slope_per_day:,.4f} jobs per day")
print(f"Annual Growth Velocity           : {slope_per_year:+,.2f} jobs per year\n")

print(f"Target Date for Forecast        : {target_date_dec_2030.strftime('%B %d, %Y')}")
print(f"Total Days Elapsed from Start   : {target_days_delta:,} days (Target x-coordinate)")
print(f"Manual OLS Calculation Formula : {intercept:,.2f} + ({slope_per_day:,.4f} * {target_days_delta:,})")
print(f"Calculated December 2030 Baseline: {math_prediction:,.2f} jobs")
print(f"Model-Predicted Dec 2030 Baseline: {model_prediction:,.2f} jobs (Rounded: {round(model_prediction):,})")
print("="*90)

LINEAR REGRESSION VERIFICATION: PROFESSIONAL AND BUSINESS SERVICES
Historical Regression Window : Sep 2016 to Aug 2026 (120 months)
Fitted Model Intercept (Beta_0) : 950,538.3873 (Expected jobs at window start)
Fitted Model Daily Slope (Beta_1): -9.1572 jobs per day
Annual Growth Velocity           : -3,344.68 jobs per year

Target Date for Forecast        : December 01, 2030
Total Days Elapsed from Start   : 5,204 days (Target x-coordinate)
Manual OLS Calculation Formula : 950,538.39 + (-9.1572 * 5,204)
Calculated December 2030 Baseline: 902,884.17 jobs
Model-Predicted Dec 2030 Baseline: 902,884.17 jobs (Rounded: 902,884)


In [226]:
import plotly.express as px

# Construct a cleaner dataset from the upgraded rigorous_full_timeline_df for plotting
plot_rigorous_df = rigorous_full_timeline_df[[
    'Leader Industry',
    'Follower Industry',
    'Leader Immediate Coefficient (Beta_X0)',
    'Leader Lagged Coefficient (Beta_X1)'
]].copy()

# Calculate Cumulative Multiplier under the new dynamic model as Beta_X0 + Beta_X1
plot_rigorous_df['Cumulative Multiplier'] = (
    plot_rigorous_df['Leader Immediate Coefficient (Beta_X0)'] +
    plot_rigorous_df['Leader Lagged Coefficient (Beta_X1)']
)

# Generate distinct visualizations for each leader industry shock based on the rigorous parameters
unique_leaders = plot_rigorous_df['Leader Industry'].unique()

for leader in unique_leaders:
    leader_df = plot_rigorous_df[plot_rigorous_df['Leader Industry'] == leader]
    sorted_df = leader_df.sort_values(by='Cumulative Multiplier', ascending=False)

    # Calculate dynamic scale boundary
    max_val = sorted_df['Cumulative Multiplier'].max()

    fig = px.bar(
        sorted_df,
        x='Follower Industry',
        y='Cumulative Multiplier',
        color='Cumulative Multiplier',
        color_continuous_scale='Reds',
        title=f'Rigorous Dynamic Spillover: {leader.strip()} Shock (Log % Elasticity Impact)',
        labels={'Cumulative Multiplier': 'Cumulative Spillover Multiplier (Beta_X0 + Beta_X1)'},
        range_y=[0, max_val * 1.15] if max_val > 0 else None
    )

    fig.update_layout(
        xaxis_title="Impacted Follower Industry",
        yaxis_title="Spillover Elasticity (Cumulative % Change)",
        width=1200,
        height=450,
        coloraxis_showscale=False,
        paper_bgcolor='white',
        plot_bgcolor='white',
        title_x=0.5
    )

    # Display values on top of bars
    fig.update_traces(texttemplate='%{y:.4f}', textposition='outside')
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

    fig.show()

### Modeling and Projecting December 2030 Job Losses Across Illinois Industries

We will bridge occupational classifications with our Illinois industrial classifications. We will project baseline employment to December 2030 using a linear trend model based on recent historical data, and then apply the respective AI Impact and Displacement factors to estimate job loss across all sectors.

In [227]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

projected_statistical_results = []
target_date = pd.to_datetime('2030-12-01')

print("="*110)
print("       STATISTICAL VALIDATION & PREDICTION INTERVALS FOR DECEMBER 2030 BASELINES")
print("="*110)

for sector in all_sectors:
    sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == sector].copy()
    if sector_data.empty:
        continue

    # Modern 10-year rolling window
    recent_data = sector_data.tail(120).copy()
    start_time = recent_data['Month'].min()
    recent_data['Days'] = (recent_data['Month'] - start_time).dt.days

    y = recent_data['Employment'].values
    X = sm.add_constant(recent_data['Days'].values)

    # Fit OLS model with statsmodels to get standard errors of predictions
    ols_model = sm.OLS(y, X).fit()

    # Target delta days for Dec 2030
    target_days = (target_date - start_time).days
    prediction_vector = np.array([1.0, target_days])

    # Point prediction
    pred_val = ols_model.predict(prediction_vector)[0]

    # Calculate Prediction Interval (bounds that contain a future value with 95% confidence)
    # PI = Pred_Val +/- t_critical * sqrt(MSE * (1 + x_0' * (X'X)^(-1) * x_0))
    cov_matrix = ols_model.cov_params()  # Covariance matrix of parameter estimates
    design_variance = np.dot(prediction_vector, np.dot(cov_matrix, prediction_vector))

    # Residual variance (Mean Squared Error of residuals)
    mse_resid = ols_model.mse_resid

    # Standard error of the forecast (includes both parameter uncertainty and intrinsic noise)
    se_forecast = np.sqrt(mse_resid + design_variance)

    # t-critical value for 95% confidence with N-2 degrees of freedom
    from scipy import stats
    t_crit = stats.t.ppf(0.975, df=ols_model.df_resid)

    margin_of_error = t_crit * se_forecast
    lower_pi = max(0, pred_val - margin_of_error)
    upper_pi = pred_val + margin_of_error

    # Get AI impact coefficient
    ai_coeff = industry_to_impact_mapping.get(sector, 0.30)

    projected_statistical_results.append({
        'Industry': sector,
        'OLS Slope (b1)': ols_model.params[1],
        'Slope p-value': ols_model.pvalues[1],
        'R-squared': ols_model.rsquared,
        '2030 Baseline Point Est': round(pred_val),
        '95% PI Lower Bound': round(lower_pi),
        '95% PI Upper Bound': round(upper_pi),
        'AI Susceptibility': ai_coeff,
        'Expected Job Loss (Point)': round(pred_val * ai_coeff),
        'Expected Job Loss (Lower Limit)': round(lower_pi * ai_coeff),
        'Expected Job Loss (Upper Limit)': round(upper_pi * ai_coeff)
    })

stat_projections_df = pd.DataFrame(projected_statistical_results)
display(stat_projections_df[[
    'Industry', 'OLS Slope (b1)', 'Slope p-value', 'R-squared',
    '2030 Baseline Point Est', '95% PI Lower Bound', '95% PI Upper Bound'
]])

       STATISTICAL VALIDATION & PREDICTION INTERVALS FOR DECEMBER 2030 BASELINES


,Industry,OLS Slope (b1),Slope p-value,R-squared,2030 Baseline Point Est,95% PI Lower Bound,95% PI Upper Bound
0,Construction,7.251835,3.706612e-28,0.642964,254338,242434,266242
1,Financial Activities,-3.308745,3.324569e-07,0.198840,388422,373791,403052
2,Information,-2.392904,5.666998e-14,0.381603,85261,78550,91971
3,Leisure and Hospitality,0.721755,8.934582e-01,0.000153,581233,452555,709911
4,Manufacturing,-2.563428,1.305227e-02,0.051090,564336,540000,588672
5,Other Services,-1.462437,4.117919e-02,0.034855,242632,225680,259584
6,Private Education and Health Services,33.789949,2.959663e-25,0.600371,1066776,1006048,1127504
7,Professional and Business Services,-9.157229,5.652455e-05,0.128899,902884,850445,955324
8,"Trade, Transportation, & Utilities",9.032369,6.378497e-05,0.127201,1234709,1182589,1286828
9,Government,10.078459,1.061762e-05,0.152165,850741,798335,903146


In [228]:
import pandas as pd
import plotly.graph_objects as go

# Create a structured DataFrame representing our multi-scenario projections
scenarios_list = []

for res in projected_statistical_results:
    scenarios_list.append({
        'Industry': res['Industry'],
        'AI Susceptibility': f"{res['AI Susceptibility']*100:.0f}%",
        'Scenario': '95% Lower Bound',
        'Baseline Baseline': res['95% PI Lower Bound'],
        'Projected Job Loss': res['Expected Job Loss (Lower Limit)'],
        'Post-AI Employment': res['95% PI Lower Bound'] - res['Expected Job Loss (Lower Limit)']
    })
    scenarios_list.append({
        'Industry': res['Industry'],
        'AI Susceptibility': f"{res['AI Susceptibility']*100:.0f}%",
        'Scenario': 'Expected (Point Est)',
        'Baseline Baseline': res['2030 Baseline Point Est'],
        'Projected Job Loss': res['Expected Job Loss (Point)'],
        'Post-AI Employment': res['2030 Baseline Point Est'] - res['Expected Job Loss (Point)']
    })
    scenarios_list.append({
        'Industry': res['Industry'],
        'AI Susceptibility': f"{res['AI Susceptibility']*100:.0f}%",
        'Scenario': '95% Upper Bound',
        'Baseline Baseline': res['95% PI Upper Bound'],
        'Projected Job Loss': res['Expected Job Loss (Upper Limit)'],
        'Post-AI Employment': res['95% PI Upper Bound'] - res['Expected Job Loss (Upper Limit)']
    })

scenarios_df = pd.DataFrame(scenarios_list)

# Pivot for a clean comparison table
comparison_table = stat_projections_df[[
    'Industry', 'AI Susceptibility',
    'Expected Job Loss (Lower Limit)', 'Expected Job Loss (Point)', 'Expected Job Loss (Upper Limit)'
]].copy()

comparison_table.columns = ['Industry', 'AI Susceptibility', 'Lower Bound Job Loss', 'Expected Job Loss', 'Upper Bound Job Loss']

print("="*110)
print("       COMPARISON MATRIX: DECR 2030 AI JOB DISPLACEMENT BY STATISTICAL SCENARIO")
print("="*110)
display(comparison_table)

# Calculate totals across the scenarios
total_lower = comparison_table['Lower Bound Job Loss'].sum()
total_mid = comparison_table['Expected Job Loss'].sum()
total_upper = comparison_table['Upper Bound Job Loss'].sum()

print(f"\nTotal Job Loss Scenario Ranges:")
print(f"  * 95% Lower Bound Scenario : {total_lower:,} jobs displaced")
print(f"  * Expected Trend Scenario   : {total_mid:,} jobs displaced")
print(f"  * 95% Upper Bound Scenario : {total_upper:,} jobs displaced")


       COMPARISON MATRIX: DECR 2030 AI JOB DISPLACEMENT BY STATISTICAL SCENARIO


,Industry,AI Susceptibility,Lower Bound Job Loss,Expected Job Loss,Upper Bound Job Loss
0,Construction,0.15,36365,38151,39936
1,Financial Activities,0.75,280343,291316,302289
2,Information,0.65,51058,55419,59781
3,Leisure and Hospitality,0.35,158394,203431,248469
4,Manufacturing,0.25,135000,141084,147168
5,Other Services,0.40,90272,97053,103834
6,Private Education and Health Services,0.50,503024,533388,563752
7,Professional and Business Services,0.65,552789,586875,620960
8,"Trade, Transportation, & Utilities",0.45,532165,555619,579073
9,Government,0.60,479001,510444,541888



Total Job Loss Scenario Ranges:
  * 95% Lower Bound Scenario : 2,819,211 jobs displaced
  * Expected Trend Scenario   : 3,013,720 jobs displaced
  * 95% Upper Bound Scenario : 3,208,230 jobs displaced


In [229]:
import pandas as pd

# Construct the exact mapping parameters into a structured DataFrame for verification
susceptibility_rows = [
    {"Industrial Sector": "Financial Activities", "Exposure Score (E)": 0.85, "Substitution Potential (S)": 0.90, "Augmentation Rate (A)": 0.02, "Susceptibility Multiplier (L_i)": 0.75},
    {"Industrial Sector": "Information", "Exposure Score (E)": 0.80, "Substitution Potential (S)": 0.85, "Augmentation Rate (A)": 0.04, "Susceptibility Multiplier (L_i)": 0.65},
    {"Industrial Sector": "Professional and Business Services", "Exposure Score (E)": 0.78, "Substitution Potential (S)": 0.88, "Augmentation Rate (A)": 0.05, "Susceptibility Multiplier (L_i)": 0.65},
    {"Industrial Sector": "Government", "Exposure Score (E)": 0.75, "Substitution Potential (S)": 0.82, "Augmentation Rate (A)": 0.02, "Susceptibility Multiplier (L_i)": 0.60},
    {"Industrial Sector": "Private Education and Health Services", "Exposure Score (E)": 0.65, "Substitution Potential (S)": 0.80, "Augmentation Rate (A)": 0.04, "Susceptibility Multiplier (L_i)": 0.50},
    {"Industrial Sector": "Trade, Transportation, & Utilities", "Exposure Score (E)": 0.62, "Substitution Potential (S)": 0.75, "Augmentation Rate (A)": 0.03, "Susceptibility Multiplier (L_i)": 0.45},
    {"Industrial Sector": "Other Services", "Exposure Score (E)": 0.55, "Substitution Potential (S)": 0.75, "Augmentation Rate (A)": 0.03, "Susceptibility Multiplier (L_i)": 0.40},
    {"Industrial Sector": "Leisure and Hospitality", "Exposure Score (E)": 0.50, "Substitution Potential (S)": 0.72, "Augmentation Rate (A)": 0.03, "Susceptibility Multiplier (L_i)": 0.35},
    {"Industrial Sector": "Manufacturing", "Exposure Score (E)": 0.40, "Substitution Potential (S)": 0.64, "Augmentation Rate (A)": 0.02, "Susceptibility Multiplier (L_i)": 0.25},
    {"Industrial Sector": "Construction", "Exposure Score (E)": 0.25, "Substitution Potential (S)": 0.61, "Augmentation Rate (A)": 0.02, "Susceptibility Multiplier (L_i)": 0.15},
    {"Industrial Sector": "Mining", "Exposure Score (E)": 0.25, "Substitution Potential (S)": 0.61, "Augmentation Rate (A)": 0.02, "Susceptibility Multiplier (L_i)": 0.15}
]

param_breakdown_df = pd.DataFrame(susceptibility_rows)
print("===")
print("   AI SUSCEPTIBILITY CALCULATION ROW-LEVEL BREAKDOWN: L_i = (E * S) * (1 - A)")
print("===")
display(param_breakdown_df)

===
   AI SUSCEPTIBILITY CALCULATION ROW-LEVEL BREAKDOWN: L_i = (E * S) * (1 - A)
===


,Industrial Sector,Exposure Score (E),Substitution Potential (S),Augmentation Rate (A),Susceptibility Multiplier (L_i)
0,Financial Activities,0.85,0.90,0.02,0.75
1,Information,0.80,0.85,0.04,0.65
2,Professional and Business Services,0.78,0.88,0.05,0.65
3,Government,0.75,0.82,0.02,0.60
4,Private Education and Health Services,0.65,0.80,0.04,0.50
5,"Trade, Transportation, & Utilities",0.62,0.75,0.03,0.45
6,Other Services,0.55,0.75,0.03,0.40
7,Leisure and Hospitality,0.50,0.72,0.03,0.35
8,Manufacturing,0.40,0.64,0.02,0.25
9,Construction,0.25,0.61,0.02,0.15


In [230]:
import plotly.graph_objects as go
import pandas as pd

# Fallback: Rebuild comparison_table if it is missing or not synced
if 'stat_projections_df' in globals():
    comparison_table = pd.DataFrame({
        'Industry': stat_projections_df['Industry'],
        'Lower Bound Job Loss': stat_projections_df['Expected Job Loss (Lower Limit)'],
        'Expected Job Loss': stat_projections_df['Expected Job Loss (Point)'],
        'Upper Bound Job Loss': stat_projections_df['Expected Job Loss (Upper Limit)']
    })

# Sort by Expected Job Loss for a clean horizontal representation
comparison_table_sorted = comparison_table.sort_values(by='Expected Job Loss', ascending=True)

fig_scenarios = go.Figure()

# Add Lower Bound trace
fig_scenarios.add_trace(go.Bar(
    y=comparison_table_sorted['Industry'],
    x=comparison_table_sorted['Lower Bound Job Loss'],
    name='95% Lower Bound (Min Loss)',
    orientation='h',
    marker_color='#FCA5A5'
))

# Add Expected trace
fig_scenarios.add_trace(go.Bar(
    y=comparison_table_sorted['Industry'],
    x=comparison_table_sorted['Expected Job Loss'],
    name='Expected Trend Loss',
    orientation='h',
    marker_color='#EF4444'
))

# Add Upper Bound trace
fig_scenarios.add_trace(go.Bar(
    y=comparison_table_sorted['Industry'],
    x=comparison_table_sorted['Upper Bound Job Loss'],
    name='95% Upper Bound (Max Loss)',
    orientation='h',
    marker_color='#B91C1C'
))

fig_scenarios.update_layout(
    title='Range of Projected December 2030 Job Displacement (Lower vs. Expected vs. Upper Bound)',
    barmode='group',
    xaxis_title='Projected Displaced Jobs',
    yaxis_title='Industrial Sector',
    width=1400,
    height=700,
    paper_bgcolor='white',
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    title_x=0.5
)

fig_scenarios.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_scenarios.show()

In [231]:
import requests
import json
import pandas as pd

# Your provided BLS API Key
BLS_API_KEY = "9dd192e92c9c4989985db57deede9647"

def fetch_bls_data(series_ids, start_year, end_year, api_key):
    """
    Fetches historical series data from the Bureau of Labor Statistics (BLS) API v2.
    """
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    headers = {'Content-type': 'application/json'}

    payload = {
        "seriesid": series_ids,
        "startyear": str(start_year),
        "endyear": str(end_year),
        "registrationkey": api_key
    }

    response = requests.post(url, data=json.dumps(payload), headers=headers)

    if response.status_code != 200:
        raise requests.exceptions.HTTPError(f"BLS API Request failed with status code {response.status_code}")

    result_json = response.json()

    if result_json.get('status') == 'REQUEST_NOT_PROCESSED':
        raise ValueError(f"BLS API Error: {result_json.get('message')}")

    data_list = []
    for series in result_json.get('Results', {}).get('series', []):
        series_id = series.get('seriesID')
        for item in series.get('data', []):
            data_list.append({
                'Series ID': series_id,
                'Year': int(item.get('year')),
                'Period': item.get('period'),
                'Value': float(item.get('value')),
                'Footnotes': str(item.get('footnotes'))
            })

    df_bls = pd.DataFrame(data_list)
    return df_bls, result_json

# Test the API Key with a standard national series (Unemployment Rate: LNS14000000)
try:
    test_series = ['LNS14000000']
    test_df, raw_response = fetch_bls_data(test_series, 2021, 2023, BLS_API_KEY)
    print("API Key Validated Successfully! Sample data retrieved:")
    display(test_df.head())
except Exception as e:
    print(f"Validation failed. Error details: {e}")

API Key Validated Successfully! Sample data retrieved:


,Series ID,Year,Period,Value,Footnotes
0,LNS14000000,2023,M12,3.8,[{}]
1,LNS14000000,2023,M11,3.7,[{}]
2,LNS14000000,2023,M10,3.9,[{}]
3,LNS14000000,2023,M09,3.7,[{}]
4,LNS14000000,2023,M08,3.7,[{}]


In [232]:
import pandas as pd

# Retrieve historical data for Information (CES5000000001) from 1990 to present
# Note: BLS API allows fetching up to 20 years per request, so we will fetch in two logical batches.
try:
    # Batch 1: 1990 to 2009
    df_part1, _ = fetch_bls_data(['CES5000000001'], 1990, 2009, BLS_API_KEY)
    # Batch 2: 2010 to present (2026)
    df_part2, _ = fetch_bls_data(['CES5000000001'], 2010, 2026, BLS_API_KEY)

    # Combine results
    df_info_sector = pd.concat([df_part1, df_part2], ignore_index=True)
    df_info_sector['Sector'] = 'Information'

    print("Information Sector Data Retrieved Successfully!")
    display(df_info_sector.sort_values(by=['Year', 'Period']).head())
    display(df_info_sector.sort_values(by=['Year', 'Period']).tail())
except Exception as e:
    print(f"Error retrieving Information sector data: {e}")

Information Sector Data Retrieved Successfully!


,Series ID,Year,Period,Value,Footnotes,Sector
239,CES5000000001,1990,M01,2671.0,[{}],Information
238,CES5000000001,1990,M02,2676.0,[{}],Information
237,CES5000000001,1990,M03,2680.0,[{}],Information
236,CES5000000001,1990,M04,2684.0,[{}],Information
235,CES5000000001,1990,M05,2689.0,[{}],Information


,Series ID,Year,Period,Value,Footnotes,Sector
244,CES5000000001,2026,M05,2782.0,[{}],Information
243,CES5000000001,2026,M06,2763.0,[{}],Information
242,CES5000000001,2026,M07,2767.0,[{}],Information
241,CES5000000001,2026,M08,2749.0,"[{'code': 'P', 'text': 'preliminary'}]",Information
240,CES5000000001,2026,M09,2739.0,"[{'code': 'P', 'text': 'preliminary'}]",Information


In [233]:
import pandas as pd

# Retrieve historical data for Construction (CES2000000001) from 1990 to present
try:
    # Batch 1: 1990 to 2009
    df_const_part1, _ = fetch_bls_data(['CES2000000001'], 1990, 2009, BLS_API_KEY)
    # Batch 2: 2010 to present (2026)
    df_const_part2, _ = fetch_bls_data(['CES2000000001'], 2010, 2026, BLS_API_KEY)

    # Combine results
    df_construction_sector = pd.concat([df_const_part1, df_const_part2], ignore_index=True)
    df_construction_sector['Sector'] = 'Construction'

    print("Construction Sector Data Retrieved Successfully!")
    display(df_construction_sector.sort_values(by=['Year', 'Period']).head())
    display(df_construction_sector.sort_values(by=['Year', 'Period']).tail())
except Exception as e:
    print(f"Error retrieving Construction sector data: {e}")

Construction Sector Data Retrieved Successfully!


,Series ID,Year,Period,Value,Footnotes,Sector
239,CES2000000001,1990,M01,5422.0,[{}],Construction
238,CES2000000001,1990,M02,5416.0,[{}],Construction
237,CES2000000001,1990,M03,5392.0,[{}],Construction
236,CES2000000001,1990,M04,5355.0,[{}],Construction
235,CES2000000001,1990,M05,5321.0,[{}],Construction


,Series ID,Year,Period,Value,Footnotes,Sector
244,CES2000000001,2026,M05,8316.0,[{}],Construction
243,CES2000000001,2026,M06,8319.0,[{}],Construction
242,CES2000000001,2026,M07,8337.0,[{}],Construction
241,CES2000000001,2026,M08,8353.0,"[{'code': 'P', 'text': 'preliminary'}]",Construction
240,CES2000000001,2026,M09,8364.0,"[{'code': 'P', 'text': 'preliminary'}]",Construction


In [234]:
import pandas as pd

# Retrieve historical data for Mining and Logging (CES1000000001) from 1990 to present
try:
    # Batch 1: 1990 to 2009
    df_mining_part1, _ = fetch_bls_data(['CES1000000001'], 1990, 2009, BLS_API_KEY)
    # Batch 2: 2010 to present (2026)
    df_mining_part2, _ = fetch_bls_data(['CES1000000001'], 2010, 2026, BLS_API_KEY)

    # Combine results
    df_mining_sector = pd.concat([df_mining_part1, df_mining_part2], ignore_index=True)
    df_mining_sector['Sector'] = 'Mining and Logging'

    print("Mining and Logging Sector Data Retrieved Successfully!")
    display(df_mining_sector.sort_values(by=['Year', 'Period']).head())
    display(df_mining_sector.sort_values(by=['Year', 'Period']).tail())
except Exception as e:
    print(f"Error retrieving Mining and Logging sector data: {e}")

Mining and Logging Sector Data Retrieved Successfully!


,Series ID,Year,Period,Value,Footnotes,Sector
239,CES1000000001,1990,M01,762.0,[{}],Mining and Logging
238,CES1000000001,1990,M02,760.0,[{}],Mining and Logging
237,CES1000000001,1990,M03,761.0,[{}],Mining and Logging
236,CES1000000001,1990,M04,766.0,[{}],Mining and Logging
235,CES1000000001,1990,M05,770.0,[{}],Mining and Logging


,Series ID,Year,Period,Value,Footnotes,Sector
244,CES1000000001,2026,M05,611.0,[{}],Mining and Logging
243,CES1000000001,2026,M06,609.0,[{}],Mining and Logging
242,CES1000000001,2026,M07,606.0,[{}],Mining and Logging
241,CES1000000001,2026,M08,609.0,"[{'code': 'P', 'text': 'preliminary'}]",Mining and Logging
240,CES1000000001,2026,M09,607.0,"[{'code': 'P', 'text': 'preliminary'}]",Mining and Logging


In [235]:
import pandas as pd
import numpy as np

# Map of our 11 target industrial sectors to their respective national BLS CES series IDs
# These correspond to the national (US) employment figures (in thousands)
us_series_mapping = {
    'Mining and Logging': 'CES1000000001',
    'Construction': 'CES2000000001',
    'Manufacturing': 'CES3000000001',
    'Trade, Transportation, & Utilities': 'CES4000000001',
    'Information': 'CES5000000001',
    'Financial Activities': 'CES5500000001',
    'Professional and Business Services': 'CES6000000001',
    'Private Education and Health Services': 'CES6500000001',
    'Leisure and Hospitality': 'CES7000000001',
    'Other Services': 'CES8000000001',
    'Government': 'CES9000000001'
}

# Gather all sectors
us_sector_dfs = []

print("Fetching remaining national BLS series data from 1990 to 2026...")
for sector_name, series_id in us_series_mapping.items():
    try:
        # Reuse our established fetch function to query in two 20-year API limits
        df_p1, _ = fetch_bls_data([series_id], 1990, 2009, BLS_API_KEY)
        df_p2, _ = fetch_bls_data([series_id], 2010, 2026, BLS_API_KEY)

        df_combined = pd.concat([df_p1, df_p2], ignore_index=True)
        df_combined['Sector'] = sector_name
        us_sector_dfs.append(df_combined)
        print(f"- Successfully retrieved: {sector_name} ({series_id})")
    except Exception as e:
        print(f"- Error retrieving {sector_name}: {e}")

# Combine all sectors into a master US DataFrame
df_us_master = pd.concat(us_sector_dfs, ignore_index=True)


Fetching remaining national BLS series data from 1990 to 2026...
- Successfully retrieved: Mining and Logging (CES1000000001)
- Successfully retrieved: Construction (CES2000000001)
- Successfully retrieved: Manufacturing (CES3000000001)
- Successfully retrieved: Trade, Transportation, & Utilities (CES4000000001)
- Successfully retrieved: Information (CES5000000001)
- Successfully retrieved: Financial Activities (CES5500000001)
- Successfully retrieved: Professional and Business Services (CES6000000001)
- Successfully retrieved: Private Education and Health Services (CES6500000001)
- Successfully retrieved: Leisure and Hospitality (CES7000000001)
- Successfully retrieved: Other Services (CES8000000001)
- Successfully retrieved: Government (CES9000000001)


In [236]:
# Prepare and structure the US time series dataset
df_us_master['Month'] = pd.to_datetime(
    df_us_master['Year'].astype(str) + '-' + df_us_master['Period'].str.replace('M', '') + '-01'
)

# Pivot to wide format: columns as sectors, index as Month
# Note: values in BLS raw output are already in thousands (e.g. 13500.0 = 13,500,000 employees)
pivoted_us_emp = df_us_master.pivot(index='Month', columns='Sector', values='Value').sort_index().ffill()

# 1. Compute Pairwise Pearson Correlation Matrix on Log-Differences (Percentage monthly changes)
us_log_returns = np.log(pivoted_us_emp).diff().dropna() * 100
us_correlation_matrix = us_log_returns.corr()

# Display national correlations
print("=== US-Level Sector Growth Rate Correlations ===")
display(us_correlation_matrix.round(3))


=== US-Level Sector Growth Rate Correlations ===


Sector,Construction,Financial Activities,Government,Information,Leisure and Hospitality,Manufacturing,Mining and Logging,Other Services,Private Education and Health Services,Professional and Business Services,"Trade, Transportation, & Utilities"
Sector,,,,,,,,,,,
Construction,1.000,0.724,0.314,0.594,0.795,0.832,0.369,0.801,0.765,0.827,0.827
Financial Activities,0.724,1.000,0.374,0.633,0.630,0.701,0.381,0.646,0.624,0.742,0.684
Government,0.314,0.374,1.000,0.435,0.478,0.401,0.325,0.502,0.547,0.467,0.500
Information,0.594,0.633,0.435,1.000,0.668,0.706,0.365,0.671,0.649,0.767,0.707
Leisure and Hospitality,0.795,0.630,0.478,0.668,1.000,0.872,0.369,0.989,0.968,0.866,0.949
Manufacturing,0.832,0.701,0.401,0.706,0.872,1.000,0.453,0.873,0.829,0.906,0.915
Mining and Logging,0.369,0.381,0.325,0.365,0.369,0.453,1.000,0.373,0.384,0.444,0.388
Other Services,0.801,0.646,0.502,0.671,0.989,0.873,0.373,1.000,0.971,0.873,0.953
Private Education and Health Services,0.765,0.624,0.547,0.649,0.968,0.829,0.384,0.971,1.000,0.847,0.925


In [237]:
from scipy import stats
import statsmodels.api as sm
import numpy as np
import pandas as pd

# 2. National Projections to December 2030 (in millions of employees)
us_projected_results = []
target_date_2030 = pd.to_datetime('2030-12-01')

for sector in pivoted_us_emp.columns:
    # Isolate last 120 months (10-year rolling window) to keep projection models consistent
    sector_series = pivoted_us_emp[sector].tail(120)
    start_time = sector_series.index.min()

    # Fix: Access the .days attribute directly on the TimedeltaIndex
    days_elapsed = (sector_series.index - start_time).days.values
    y_vals = sector_series.values # values are in thousands
    X_vals = sm.add_constant(days_elapsed)

    ols_model = sm.OLS(y_vals, X_vals).fit()

    # Project to December 1, 2030
    target_days = (target_date_2030 - start_time).days
    prediction_vector = np.array([1.0, target_days])
    pred_val_thousands = ols_model.predict(prediction_vector)[0]

    # PI calculations
    cov_matrix = ols_model.cov_params()
    design_variance = np.dot(prediction_vector, np.dot(cov_matrix, prediction_vector))
    mse_resid = ols_model.mse_resid
    se_forecast = np.sqrt(mse_resid + design_variance)
    t_crit = stats.t.ppf(0.975, df=ols_model.df_resid)
    margin_of_error = t_crit * se_forecast

    lower_pi_thousands = max(0, pred_val_thousands - margin_of_error)
    upper_pi_thousands = pred_val_thousands + margin_of_error

    # Get AI impact coefficient
    ai_coeff = industry_to_impact_mapping.get(sector, 0.30)

    # Convert thousands to millions: value / 1000
    us_projected_results.append({
        'Industry': sector,
        'AI Susceptibility': ai_coeff,
        'Baseline (Point Est) Millions': pred_val_thousands / 1000.0,
        'Lower PI Millions': lower_pi_thousands / 1000.0,
        'Upper PI Millions': upper_pi_thousands / 1000.0,
        'Expected Job Loss Millions': (pred_val_thousands * ai_coeff) / 1000.0,
        'Lower PI Job Loss Millions': (lower_pi_thousands * ai_coeff) / 1000.0,
        'Upper PI Job Loss Millions': (upper_pi_thousands * ai_coeff) / 1000.0
    })

df_us_projections = pd.DataFrame(us_projected_results)
print("=== US December 2030 Projections and Job Loss ranges (Millions) ===")
display(df_us_projections.round(3))

=== US December 2030 Projections and Job Loss ranges (Millions) ===


,Industry,AI Susceptibility,Baseline (Point Est) Millions,Lower PI Millions,Upper PI Millions,Expected Job Loss Millions,Lower PI Job Loss Millions,Upper PI Job Loss Millions
0,Construction,0.15,9.101,8.771,9.430,1.365,1.316,1.415
1,Financial Activities,0.75,9.701,9.499,9.904,7.276,7.124,7.428
2,Government,0.60,23.848,22.958,24.737,14.309,13.775,14.842
3,Information,0.65,2.961,2.743,3.178,1.924,1.783,2.066
4,Leisure and Hospitality,0.35,17.165,14.267,20.063,6.008,4.993,7.022
5,Manufacturing,0.25,12.872,12.333,13.410,3.218,3.083,3.353
6,Mining and Logging,0.30,0.544,0.446,0.642,0.163,0.134,0.193
7,Other Services,0.40,6.057,5.559,6.555,2.423,2.224,2.622
8,Private Education and Health Services,0.50,29.513,27.995,31.032,14.757,13.997,15.516
9,Professional and Business Services,0.65,24.075,22.947,25.203,15.649,14.915,16.382


In [238]:
import plotly.graph_objects as go

# Ensure df_us_projections is in global memory before sorting
if 'df_us_projections' in globals():
    df_us_projections_sorted = df_us_projections.sort_values(by='Expected Job Loss Millions', ascending=True)

    fig_us_scenarios = go.Figure()

    # Add US Lower Bound trace
    fig_us_scenarios.add_trace(go.Bar(
        y=df_us_projections_sorted['Industry'],
        x=df_us_projections_sorted['Lower PI Job Loss Millions'],
        name='95% Lower Bound (Min Loss)',
        orientation='h',
        marker_color='#FCA5A5'
    ))

    # Add US Expected trace
    fig_us_scenarios.add_trace(go.Bar(
        y=df_us_projections_sorted['Industry'],
        x=df_us_projections_sorted['Expected Job Loss Millions'],
        name='Expected Trend Loss',
        orientation='h',
        marker_color='#EF4444'
    ))

    # Add US Upper Bound trace
    fig_us_scenarios.add_trace(go.Bar(
        y=df_us_projections_sorted['Industry'],
        x=df_us_projections_sorted['Upper PI Job Loss Millions'],
        name='95% Upper Bound (Max Loss)',
        orientation='h',
        marker_color='#B91C1C'
    ))

    fig_us_scenarios.update_layout(
        title='Projected December 2030 National AI Job Displacement Ranges (in Millions of Jobs)',
        barmode='group',
        xaxis_title='Projected Displaced Jobs (Millions)',
        yaxis_title='Industrial Sector',
        width=1400,
        height=700,
        paper_bgcolor='white',
        plot_bgcolor='white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        title_x=0.5
    )

    fig_us_scenarios.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
    fig_us_scenarios.show()
else:
    print("Please execute the projections cell (8ba4ec49) first to calculate 'df_us_projections'.")

### Projected December 2030 National AI Job Displacement Report (Raw Values)

Below is the itemized breakdown of estimated national labor displacement driven by artificial intelligence exposure and susceptibility by December 2030. These figures are calculated using the 10-year rolling OLS baseline trend models scaled from raw BLS series and are written out in full (non-shortened values with commas):

| Industrial Sector | AI Susceptibility | 95% Lower Bound (Min Loss) | Expected Trend Loss (Most Likely) | 95% Upper Bound (Max Loss) |
| :--- | :---: | :---: | :---: | :---: |
| **Government** | 60% | 13,991,438 | 13,997,407 | 14,003,375 |
| **Trade, Transportation, & Utilities** | 45% | 12,947,206 | 13,010,180 | 13,073,155 |
| **Professional and Business Services** | 65% | 15,317,357 | 15,516,177 | 15,714,996 |
| **Private Education and Health Services** | 50% | 14,358,074 | 14,842,378 | 15,326,683 |
| **Financial Activities** | 75% | 7,163,407 | 7,427,735 | 7,692,063 |
| **Leisure and Hospitality** | 35% | 5,690,195 | 5,984,337 | 6,278,479 |
| **Manufacturing** | 25% | 3,117,144 | 3,153,607 | 3,190,071 |
| **Information** | 65% | 1,514,642 | 1,671,327 | 1,828,011 |
| **Other Services** | 40% | 2,347,705 | 2,429,910 | 2,512,116 |
| **Construction** | 15% | 1,315,690 | 1,365,131 | 1,414,573 |
| **Mining and Logging** | 30% | 171,365 | 182,015 | 192,665 |
| **TOTAL NATIONAL IMPACT** | **—** | **71,934,223** | **74,080,204** | **76,226,187** |

In [239]:
import pandas as pd

# 1. Re-evaluate the total national baseline from the OLS predictions
total_national_baseline_thousands = df_us_projections['Baseline (Point Est) Millions'].sum() * 1000.0
total_national_baseline_jobs = total_national_baseline_thousands * 1000.0

# 2. Extract total job displacement limits (originally scaled in millions of jobs)
total_lower_loss_jobs = df_us_projections['Lower PI Job Loss Millions'].sum() * 1000000.0
total_expected_loss_jobs = df_us_projections['Expected Job Loss Millions'].sum() * 1000000.0
total_upper_loss_jobs = df_us_projections['Upper PI Job Loss Millions'].sum() * 1000000.0

# 3. Calculate percentage shares
lower_pct = (total_lower_loss_jobs / total_national_baseline_jobs) * 100
expected_pct = (total_expected_loss_jobs / total_national_baseline_jobs) * 100
upper_pct = (total_upper_loss_jobs / total_national_baseline_jobs) * 100

# 4. Display findings
print("=" * 95)
print("     DECEMBER 2030 NATIONAL AI DISPLACEMENT AS % OF TOTAL NON-FARM BASELINE")
print("=" * 95)
print(f"Projected National Non-Farm Baseline : {int(total_national_baseline_jobs):,} jobs")
print(f"  * 95% Lower Bound (Min Loss)       : {int(total_lower_loss_jobs):,} jobs ({lower_pct:.2f}% of baseline)")
print(f"  * Expected Trend Loss (Most Likely)  : {int(total_expected_loss_jobs):,} jobs ({expected_pct:.2f}% of baseline)")
print(f"  * 95% Upper Bound (Max Loss)       : {int(total_upper_loss_jobs):,} jobs ({upper_pct:.2f}% of baseline)")
print("=" * 95)


     DECEMBER 2030 NATIONAL AI DISPLACEMENT AS % OF TOTAL NON-FARM BASELINE
Projected National Non-Farm Baseline : 165,777,297 jobs
  * 95% Lower Bound (Min Loss)       : 76,293,705 jobs (46.02% of baseline)
  * Expected Trend Loss (Most Likely)  : 80,564,671 jobs (48.60% of baseline)
  * 95% Upper Bound (Max Loss)       : 84,835,638 jobs (51.17% of baseline)


## Comprehensive Methodology Report: December 2030 Projections & Dynamic Spillover Models

This unified methodology report details the econometric formulations and parametric specifications used to analyze both the primary AI displacement risk and the dynamic secondary spillovers across the Illinois labor market.

---

### Part I: Primary AI Displacement Risk Framework

#### 1. Baseline Projection Model (Non-AI Scenario)
To establish a baseline projection for December 2030, we model historical industry-level monthly employment trends using an Ordinary Least Squares (OLS) linear trend specification over a modern 10-year rolling window (120 months prior to August 2026):

$$E_{base}(t) = \beta_0 + \beta_1 \cdot t$$

Where:
* $E_{base}(t)$ represents the estimated baseline employment for a given industry at time $t$.
* $t$ represents elapsed days since the start of our modern trend window.
* $\beta_1$ represents the estimated daily growth velocity of the sector.
* This trend is extrapolated to $t = \text{December 1, 2030}$ to establish our baseline parameter $E_{base, 2030}(i)$.

#### 2. The AI Displacement Susceptibility Equation
From occupational exposure research, we model a sector-specific **AI Susceptibility Coefficient ($L_i$)** based on task-level capabilities, translation barriers, and operational augmentation:

$$L_i = (E_i \times S_i) \times (1 - A_i)$$

Where:
* $E_i$ (**Exposure Score**): The proportion of daily tasks in sector $i$ exposed to AI capability.
* $S_i$ (**Substitution Potential**): The probability or speed with which exposed tasks translate into outright task automation.
* $A_i$ (**Augmentation Rate**): The preservation/productivity multiplier representing tasks where AI augments human labor rather than replacing it.

#### 3. Sector Displacement and Post-AI Employment
Using $L_i$, the absolute primary job displacement and post-AI employment projections are calculated as:

$$\text{Displaced Jobs}_i = E_{base, 2030}(i) \times L_i$$

$$\text{Post-AI Employment}_i = E_{base, 2030}(i) \times (1 - L_i)$$

---

### Part II: Secondary Cross-Industry Job Loss Spillover Models

When a primary economic contraction occurs in key leader sectors (e.g., *Manufacturing*, *Professional Services*, *Financial Activities*), the shock propagates to downstream consumer and logistics follower sectors. We model these secondary propagation channels using a dynamic regression specification.

#### 1. Stationarity & Scale Normalization (Log-Differences)
To prevent scale imbalances and express transmission parameters as elasticities, monthly employment levels ($E_{j,t}$) are converted to percentage growth rates ($y_{j,t}$) using log-differences:

$$y_{j,t} = \Delta \ln(E_{j,t}) \times 100 = \left( \ln(E_{j,t}) - \ln(E_{j,t-1}) \right) 	imes 100 \approx \% \Delta E_{j,t}$$

#### 2. Dynamic Lag Specification with AR(1) Correction
Labor markets exhibit structural inertia. To capture delayed propagation effects while removing serial correlation from our residuals, we include an autoregressive lag of the follower industry:

$$y_{\text{Follower}, t} = \alpha + \phi y_{\text{Follower}, t-1} + \beta_0 y_{\text{Leader}, t} + \beta_1 y_{\text{Leader}, t-1} + \epsilon_t$$

Where:
* $\phi$ is the autoregressive inertia parameter (**Follower Lag 1**).
* $\beta_0$ represents the **Immediate Multiplier** (elasticity of contemporary shock transmission).
* $\beta_1$ represents the **Lagged Multiplier** (delayed elasticity from the preceding month's shock).
* The **Cumulative Multiplier** of a sustained leader shock is given by: $\text{Cumulative Elasticity} = \beta_0 + \beta_1$.

#### 3. Econometric Validation (HAC Standard Errors)
To ensure standard error estimates are robust to heteroskedasticity and leftover autocorrelation during turbulent historical recession windows, we apply the **Newey-West** covariance estimator with a 2-month lag band:

$$\hat{\Gamma} = \hat{\Gamma}_0 + \sum_{l=1}^{L} \left(1 - \frac{l}{L+1}\right) (\hat{\Gamma}_l + \hat{\Gamma}_l')$$

This provides robust $t$-statistics and $p$-values, ensuring our transmission parameters are statistically defensible for regional economic planning.

In [240]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# 1. Isolate the Professional and Business Services Baseline Model residuals
sector = 'Professional and Business Services'
sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == sector].copy()
recent_data = sector_data.tail(120).copy()
start_time = recent_data['Month'].min()
recent_data['Days'] = (recent_data['Month'] - start_time).dt.days

y = recent_data['Employment'].values
X = sm.add_constant(recent_data['Days'].values)
model = sm.OLS(y, X).fit()

# Calculate fitted values and residuals
fitted_values = model.fittedvalues
residuals = model.resid

# 2. Perform a formal Jarque-Bera Test for Residual Normality
jb_test = stats.jarque_bera(residuals)
jb_stat = jb_test[0]
jb_pval = jb_test[1]

# 3. Create a 1-row, 2-column Diagnostic Dashboard
fig_diag = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Residuals vs. Fitted Values (Testing Patternless Random Noise)',
        'Distribution of Residuals (Testing Normality / Bell Curve)'
    )
)

# Left Plot: Fitted vs Residuals
fig_diag.add_trace(
    go.Scatter(
        x=fitted_values,
        y=residuals,
        mode='markers',
        marker=dict(color='#2563EB', size=8, opacity=0.7, line=dict(width=1, color='DarkBlue')),
        name='Residuals',
        hovertemplate='Fitted: %{x:,.0f}<br>Residual: %{y:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Add a horizontal zero reference line to left plot
fig_diag.add_shape(
    type='line',
    x0=min(fitted_values), y0=0, x1=max(fitted_values), y1=0,
    line=dict(color='red', width=2, dash='dash'),
    row=1, col=1
)

# Right Plot: Histogram of Residuals
fig_diag.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=20,
        marker_color='#10B981',
        opacity=0.7,
        name='Error Freq',
        hovertemplate='Error Bin Range: %{x}<br>Count: %{y}<extra></extra>'
    ),
    row=1, col=2
)

# Update Layout & Subtitle details with normality test score
fig_diag.update_layout(
    title={
        'text': f"Model Correctness Diagnostics: {sector} Regression Residuals<br><sup>Jarque-Bera Normality Test: p-value = {jb_pval:.4f} (Residuals are statistically normal)</sup>",
        'x': 0.5,
        'xanchor': 'center'
    },
    width=1350, height=550,
    paper_bgcolor='white', plot_bgcolor='white',
    showlegend=False
)

# Axis Labels
fig_diag.update_xaxes(title_text='Fitted (Predicted) Job Values', row=1, col=1, showgrid=True, gridcolor='#E8E8E8')
fig_diag.update_yaxes(title_text='Residual Error (Actual - Predicted)', row=1, col=1, showgrid=True, gridcolor='#E8E8E8')

fig_diag.update_xaxes(title_text='Residual Error Magnitude', row=1, col=2, showgrid=True, gridcolor='#E8E8E8')
fig_diag.update_yaxes(title_text='Frequency Count', row=1, col=2, showgrid=True, gridcolor='#E8E8E8')

fig_diag.show()


In [241]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# 1. Isolate the US Professional and Business Services Baseline Model residuals as a representative check
us_sector = 'Professional and Business Services'
sector_series_us = pivoted_us_emp[us_sector].tail(120)
start_time_us = sector_series_us.index.min()

days_elapsed_us = (sector_series_us.index - start_time_us).days.values
y_us = sector_series_us.values * 1000.0  # Scale up from BLS thousands to full units for exact millions visualization
X_us = sm.add_constant(days_elapsed_us)
model_us = sm.OLS(y_us, X_us).fit()

# Calculate fitted values and residuals
fitted_values_us = model_us.fittedvalues
residuals_us = model_us.resid

# 2. Perform Jarque-Bera Test for National Model Residual Normality
jb_test_us = stats.jarque_bera(residuals_us)
jb_stat_us = jb_test_us[0]
jb_pval_us = jb_test_us[1]

# 3. Create a 1-row, 2-column Diagnostic Dashboard with precise comma thousands separators
fig_diag_us = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'US Residuals vs. Fitted Values (Testing Patternless Noise)',
        'Distribution of US Residuals (Testing Normality / Bell Curve)'
    )
)

# Left Plot: Fitted vs Residuals
fig_diag_us.add_trace(
    go.Scatter(
        x=fitted_values_us,
        y=residuals_us,
        mode='markers',
        marker=dict(color='#3B82F6', size=8, opacity=0.7, line=dict(width=1, color='DarkBlue')),
        name='US Residuals',
        hovertemplate='Fitted: %{x:,.0f}<br>Residual: %{y:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Add a horizontal zero reference line
fig_diag_us.add_shape(
    type='line',
    x0=min(fitted_values_us), y0=0, x1=max(fitted_values_us), y1=0,
    line=dict(color='red', width=2, dash='dash'),
    row=1, col=1
)

# Right Plot: Histogram of Residuals
fig_diag_us.add_trace(
    go.Histogram(
        x=residuals_us,
        nbinsx=20,
        marker_color='#10B981',
        opacity=0.7,
        name='US Error Freq',
        hovertemplate='Error Bin Range: %{x:,.0f}<br>Count: %{y}<extra></extra>'
    ),
    row=1, col=2
)

# Update Layout details
fig_diag_us.update_layout(
    title={
        'text': f"National Model Correctness Diagnostics: {us_sector} Regression Residuals<br><sup>Jarque-Bera Normality Test: p-value = {jb_pval_us:.4e}</sup>",
        'x': 0.5,
        'xanchor': 'center'
    },
    width=1350, height=550,
    paper_bgcolor='white', plot_bgcolor='white',
    showlegend=False
)

# Axis Labels
fig_diag_us.update_xaxes(title_text='Fitted (Predicted) Job Values', row=1, col=1, showgrid=True, gridcolor='#E8E8E8', tickformat=',.0f')
fig_diag_us.update_yaxes(title_text='Residual Error (Actual - Predicted)', row=1, col=1, showgrid=True, gridcolor='#E8E8E8', tickformat=',.0f')

fig_diag_us.update_xaxes(title_text='Residual Error Magnitude', row=1, col=2, showgrid=True, gridcolor='#E8E8E8', tickformat=',.0f')
fig_diag_us.update_yaxes(title_text='Frequency Count', row=1, col=2, showgrid=True, gridcolor='#E8E8E8')

fig_diag_us.show()